# PDS 37 with VLTI/GRAVITY: building up a model, one step at a time

Using a single exposure, this notebook tests increasingly complex geometric models
against the GRAVITY data with [PMOIRED](https://github.com/amerand/PMOIRED):

1. a single star plus extended emission,
2. a binary of two uniform disks (companion position found by a grid search),
3. limb-darkened disks with Brackett-γ line emission,
4. adding azimuthal brightness variations to the companion's disk.

Model quality is judged with the reduced χ² and by eye from the fit residuals.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pmoired

sys.path.insert(0, str(Path.cwd().parent))  # make the local `pds37` package importable
from pds37 import config

plt.rcParams["figure.dpi"] = 80
from pds37.binary import binary_grid_search, image_window, BRG_LINE_PRIORS

FILE = str(config.gravity_files()[0])  # any single exposure
chi2 = {}  # reduced chi2 of each model, for the summary at the end

## 1. Single star + fully resolved emission

The data are binned by 20 spectral channels (R ≈ 200) since continuum geometry does not need full spectral
resolution, and fitting is much faster.

In [ ]:
oi = pmoired.OI(FILE, insname=config.GRAVITY_INSNAME, binning=20, verbose=False)
oi.setupFit({"obs": ["|V|", "DPHI", "NFLUX", "T3PHI"], "wl ranges": [config.K_BAND]})

single = {"star,ud": 0.7, "star,f": 1,   # uniform disk, diameter in mas
          "ext,f": 1}                     # fully resolved component (no size given)
oi.doFit(single, doNotFit=["star,f"], verbose=1)
chi2["single star + extended"] = oi.bestfit["chi2"]
oi.show(imFov=20)

A point-symmetric model can only produce closure phases of 0° or 180°. If the measured `T3PHI` clearly
differs from that, a binary is the natural next model.

## 2. Binary of two uniform disks

The companion's position is found with a **grid search**: a fit is started from every point of a 3 mas grid
out to 50 mas, and the deepest χ² minimum is kept. This avoids getting stuck in a local minimum, which is easy for
binary models.

In [ ]:
oi.setupFit({"obs": ["|V|", "T3PHI"], "wl ranges": [config.K_BAND]})

grid_best = binary_grid_search(oi, max_sep=50, step=3,
                               model={"A,ud": 0.5, "B,ud": 0.5, "B,f": 0.9})
oi.showGrid(interpolate=True, logV=1, cmap="gist_stern")

Starting from the grid solution, all parameters (including both disk diameters) are now fitted together:

In [ ]:
oi.doFit(grid_best, doNotFit=[], verbose=1)
binary = dict(oi.bestfit["best"])
chi2["binary, uniform disks"] = oi.bestfit["chi2"]
oi.show(**image_window(binary))

## 3. Limb-darkened disks with Brackett-γ emission

Back at full spectral resolution, around Brγ only. Each component gets a disk with a `µ^0.5` limb-darkening
profile and its own Lorentzian emission line. The flux continuum is modelled by a 2nd-order polynomial.
The companion position is held fixed at the binary solution from step 2, and the primary's flux is fixed to 1
(otherwise it is fully degenerate with the line strengths).

In [ ]:
oi = pmoired.OI(FILE, insname=config.GRAVITY_INSNAME, verbose=False)
oi.setupFit({"obs": ["|V|", "T3PHI", "NFLUX"],
             "wl ranges": [config.BRG_WINDOW],
             "NFLUX order": 2})

line_model = {
    "A,f": 1, "A,x": 0, "A,y": 0,
    "A,diam": 5.0, "A,profile": "$MU**0.5",
    "B,f": binary["B,f"], "B,x": binary["B,x"], "B,y": binary["B,y"],
    "B,diam": 50.0, "B,profile": "$MU**0.5",
    "A,line_brg_wl0": 2.166, "A,line_brg_f": 1.0, "A,line_brg_lorentzian": 1.0,
    "B,line_brg_wl0": 2.166, "B,line_brg_f": 0.0, "B,line_brg_lorentzian": 1.0,
}
oi.doFit(line_model, prior=BRG_LINE_PRIORS, doNotFit=["A,f", "A,x", "A,y", "B,x", "B,y"], verbose=1)
chi2["limb-darkened binary + Brγ (Brγ window only)"] = oi.bestfit["chi2"]
oi.show(**image_window(oi.bestfit["best"]))

## 4. Azimuthal brightness variations

To give the model more freedom to match the closure phases, the companion's disk is allowed cosine brightness
modulations around its azimuth (orders 1–3), inclined and rotated on the sky, and both components are smoothed
with a small Gaussian spatial kernel. This uses the full K band again, binned by 15 channels.

In [ ]:
oi = pmoired.OI(FILE, insname=config.GRAVITY_INSNAME, binning=15, verbose=False)
oi.setupFit({"obs": ["|V|", "T3PHI"], "wl ranges": [config.K_BAND]})

asym_model = {
    "A,f": 1, "A,x": 0, "A,y": 0, "A,diam": 5.0, "A,profile": "$MU**0.5",
    "B,f": binary["B,f"], "B,x": binary["B,x"], "B,y": binary["B,y"],
    "B,diam": 50.0, "B,profile": "$MU**0.5",
    "B,projang": -20, "B,incl": 60,
    "A,spatial kernel": 0.2, "B,spatial kernel": 0.2,
}
orders = [1, 2, 3]
for n in orders:
    asym_model[f"B,az amp{n}"] = 1 / len(orders)
    asym_model[f"B,az projang{n}"] = 30 * n

oi.doFit(asym_model, doNotFit=["A,x", "A,y", "A,f"], verbose=1)
chi2["binary + azimuthal modulation"] = oi.bestfit["chi2"]
oi.show(**image_window(oi.bestfit["best"]))

If the added parameters (modulation amplitudes and angles, inclination, kernel sizes) come out with uncertainties
larger than their values, the data do not constrain this extra structure and the plain binary should be preferred.

## Summary

Reduced χ² is only directly comparable between models fitted to the **same data selection**:
steps 1–2 use the binned K band, step 3 uses the Brγ window at full resolution, and step 4 uses the K band binned by 15.

In [ ]:
pd.Series(chi2, name="reduced χ²").round(2).to_frame()

**Next steps**
- Fit short line-free stretches of continuum first, then add the line region.
- Repeat the grid search and refinement on every epoch to see which give well-constrained solutions
  (done in `03_gravity_binary_fits_by_epoch.ipynb`).